In [70]:
import bs4
import requests
import pandas as pd
import os
from urllib.parse import urljoin
import re
import copy

def getInfoBox(soup, url):
    table = soup.select_one('table.infobox')
    rows = table.find_all('tr')

    data = []
    current_header = "General"

    for row in rows:
        # Check if this row is an infobox header
        header_cell = row.find('th', class_='infobox-header')
        if header_cell:
            current_header = header_cell.get_text(strip=True)
            continue
        
        th = row.find('th')
        td = row.find('td', class_='infobox-data')
        
        if th and td:
            label = th.get_text(strip=True)
            
            list_items = td.find_all('li')
            containers = list_items if list_items else [td]
            
            values_list = []
            links_list = []
            citations_list = []
            
            for container_orig in containers:
                # Make a deep copy to avoid destroying citations in the original soup object
                # if this cell is executed multiple times!
                container = copy.copy(container_orig)
                
                # Extract citations: Look for sup tags that contain bracketed numbers/letters or have class reference
                cites = []
                for sup in container.find_all('sup'):
                    cls = sup.get('class', [])
                    if 'reference' in cls or 'cite_ref' in cls or (sup.get_text() and re.match(r'^\[\w+\]$', sup.get_text().strip())):
                        cites.append(sup.get_text(strip=True))
                        sup.decompose()
                
                # Extract links, ignoring units (heuristic: ignoring short strings or numbers)
                lnks = []
                for a in container.find_all('a'):
                    link_text = a.get_text(strip=True)
                    link_href = a.get('href', '')
                    # Simple heuristic to exclude typical unit/symbol links
                    if len(link_text) > 2 and not any(char.isdigit() for char in link_text):
                        if link_href:
                            full_url = urljoin(url, link_href)
                            lnks.append(full_url)
                
                # Re-get the value after decomposing citations in the temporary copy
                # Also remove parenthetical text which usually contains redundant conversions/units
                value_clean = container.get_text(separator=' ', strip=True)
                #value_clean = re.sub(r'\s*\([^)]*\)', '', value_clean).strip()
                
                # If after stripping units it becomes empty, we might want to skip it, 
                # but we shouldn't lose the citations if it had any.
                # We can merge its citations into the previous item in the list.
                if not value_clean:
                    if cites and citations_list:
                        # Append these citations to the previous item's citations
                        prev_cites = citations_list[-1]
                        if prev_cites:
                            citations_list[-1] = prev_cites + ", " + ", ".join(cites)
                        else:
                            citations_list[-1] = ", ".join(cites)
                    continue
                    
                values_list.append(value_clean)
                links_list.append(", ".join(lnks) if lnks else None)
                citations_list.append(", ".join(cites) if cites else None)
                
            if not values_list:
                continue
            
            value_out = values_list if list_items else (values_list[0] if values_list else None)
            links_out = links_list if list_items else (links_list[0] if links_list else None)
            cites_out = citations_list if list_items else (citations_list[0] if citations_list else None)
            
            # Collect links from the row label (<th>) as a list of absolute URLs
            label_links = [
                urljoin(url, a.get('href'))
                for a in th.find_all('a', href=True)
                if a.get('href')
            ]
            # Optional de-dup while preserving order
            label_links = list(dict.fromkeys(label_links))

            data.append({
                'Metadata': current_header,
                'Label': label,
                'Label Links': label_links,
                'Value': value_out,
                'Links': links_out,
                'Citations': cites_out
            })

    df = pd.DataFrame(data)
    df.set_index(['Metadata', 'Label'], inplace=True)
    return df

def savePageAndInfoBox(url, file_name, data_dir):

    # make data dir
    os.makedirs(data_dir, exist_ok=True)
    
    # soup
    headers = {"User-Agent": "garrettstoll.info/1.0 (garrett@garrettstoll.info)"}
    response = requests.get(url, headers=headers)
    soup = bs4.BeautifulSoup(response.text, 'html.parser')

    # save page
    with open(f'{data_dir}/{file_name}_wiki.html', 'w') as f:
        f.write(soup.prettify())
    
    # save infoBox
    infoBox = getInfoBox(soup, url)
    infoBox.to_csv(f'{data_dir}/{file_name}_infoBox_wiki.csv')

    return infoBox

#savePageAndInfoBox("https://en.wikipedia.org/wiki/Solar_System", 'solarsystem', 'data/Sol')
savePageAndInfoBox("https://en.wikipedia.org/wiki/Sun", 'sun', 'data/tmp')
    

Label Links  \
Category                 Label                                                                                
General                  Names                                                                           []   
                         Adjectives                                                                      []   
Observation data         Mean distance from Earth                                                        []   
                         Visual brightness                [https://en.wikipedia.org/wiki/Apparent_magnit...   
                         Absolute magnitude               [https://en.wikipedia.org/wiki/Absolute_magnit...   
                         Spectral classification          [https://en.wikipedia.org/wiki/Spectral_classi...   
                         Metallicity                            [https://en.wikipedia.org/wiki/Metallicity]   
                         Angular size                          [https://en.wikipedia.org/wiki/Angular_size]   
Orbital characteristics  Mean distance fromMilky Waycore          [https://en.wikipedia.org/wiki/Milky_Way]   
                         Galactic period                      [https://en.wikipedia.org/wiki/Galactic_year]   
                         Velocity                                  [https://en.wikipedia.org/wiki/Velocity]   
                         Obliquity                                [https://en.wikipedia.org/wiki/Obliquity]   
                         Right ascensionNorth pole          [https://en.wikipedia.org/wiki/Right_ascension]   
                         Declinationof North pole               [https://en.wikipedia.org/wiki/Declination]   
                         Siderealrotation period             [https://en.wikipedia.org/wiki/Solar_rotation]   
                         Equatorial rotation velocity                                                    []   
Physical characteristics Equatorialradius                            [https://en.wikipedia.org/wiki/Radius]   
                         Flattening                              [https://en.wikipedia.org/wiki/Flattening]   
                         Surface area                          [https://en.wikipedia.org/wiki/Surface_area]   
                         Volume                                      [https://en.wikipedia.org/wiki/Volume]   
                         Mass                                          [https://en.wikipedia.org/wiki/Mass]   
                         Averagedensity                             [https://en.wikipedia.org/wiki/Density]   
                         Age                                                                             []   
                         Equatorialsurface gravity          [https://en.wikipedia.org/wiki/Surface_gravity]   
                         Moment of inertia factor         [https://en.wikipedia.org/wiki/Moment_of_inert...   
                         Surfaceescape velocity             [https://en.wikipedia.org/wiki/Escape_velocity]   
                         Temperature                                                                     []   
                         Luminosity                              [https://en.wikipedia.org/wiki/Luminosity]   
                         Colour(B-V)                            [https://en.wikipedia.org/wiki/Color_index]   
                         Meanradiance                              [https://en.wikipedia.org/wiki/Radiance]   
                         Photospherecomposition by mass         [https://en.wikipedia.org/wiki/Photosphere]   

                                                                                                      Value  \
Category                 Label                                                                                
General                  Names                                                      Sun, Sol , Sól , Helios   
                         Adjectives                                                                

In [82]:
lst = None

# list object size (container only)
print(lst.__sizeof__(), "bytes")

16 bytes


In [108]:
a = pd.DataFrame({('Metadata', 'PageName'): ['Sun'] * 3})
a = pd.concat([a.reset_index(drop=True), df.T.reset_index()], axis=1)
a.columns = pd.MultiIndex.from_tuples([('Metadata', 'RowType') if col == ('index', '') else col for col in a.columns])
a

Metadata                         General  \
  PageName    RowType                  Age   
0      Sun      Value  4.568 billion years   
1      Sun      Links                 None   
2      Sun  Citations                  [b]   

                                                      \
                                            Location   
0  [Local Interstellar Cloud, Local Bubble, Orion...   
1  [https://en.wikipedia.org/wiki/Local_Interstel...   
2                             [None, [1], None, [2]]   

                                                      \
                                        Nearest star   
0                 [Proxima Centauri, Alpha Centauri]   
1  [https://en.wikipedia.org/wiki/Proxima_Centaur...   
2                                     [[D 1], [D 2]]   

                          Population  \
                               Stars   
0                                Sun   
1  https://en.wikipedia.org/wiki/Sun   
2                               None   

                                                      \
                                             Planets   
0  [Mercury, Venus, Earth, Mars, Jupiter, Saturn,...   
1  [https://en.wikipedia.org/wiki/Mercury_(planet...   
2   [None, None, None, None, None, None, None, None]   

                                                                              \
                                  Knowndwarf planets Knownnatural satellites   
0  [Ceres, Orcus, Pluto, Haumea, Quaoar, Makemake...                     758   
1  [https://en.wikipedia.org/wiki/Ceres_(dwarf_pl...                    None   
2  [None, None, None, None, None, None, None, Non...                   [D 3]   

                      ...   Planetary system             \
  Knownminor planets  ... Star spectral type Frost line   
0          1,462,402  ...                G2V      ~5 AU   
1               None  ...               None       None   
2              [D 4]  ...               None        [5]   

                                                                           \
      Semi-major axisof outermost planet Kuiper cliff          Heliopause   
0                               30.07 AU     50–70 AU  detected at 120 AU   
1  https://en.wikipedia.org/wiki/Neptune         None                None   
2                                  [D 5]     [3], [4]                 [6]   

                                   Orbit aboutGalactic Center  \
          Hill sphere Invariable-to-galactic planeinclination   
0  178,000–227,000 AU                   ~60°, to the ecliptic   
1                None                                    None   
2            [7], [8]                                     [c]   

                                            \
  Distance toGalactic Center Orbital speed   
0           24,000–28,000 ly  720,000 km/h   
1                       None          None   
2                        [9]          [10]   

                                               
                               Orbital period  
0                          ~230 million years  
1  https://en.wikipedia.org/wiki/Million_year  
2                                        [10]  

[3 rows x 21 columns]

# Planets

In [110]:
url = "https://en.wikipedia.org/wiki/Solar_System"
headers = {"User-Agent": "garrettstoll.info/1.0 (garrett@garrettstoll.info)"}
response = requests.get(url, headers=headers)
soup = bs4.BeautifulSoup(response.text, 'html.parser')
infobox = getInfoBox(soup, url)
infobox.loc[('Population', 'Planets')]['Value']

['Mercury', 'Venus', 'Earth', 'Mars', 'Jupiter', 'Saturn', 'Uranus', 'Neptune']

In [112]:
pd.concat([system, df], axis=0, join='outer', ignore_index=True, sort=False)

,Value,Links,Citations
0,4.568 billion years,None,[b]
1,"[Local Interstellar Cloud, Local Bubble, Orion...",[https://en.wikipedia.org/wiki/Local_Interstel...,"[None, [1], None, [2]]"
2,"[Proxima Centauri, Alpha Centauri]",[https://en.wikipedia.org/wiki/Proxima_Centaur...,"[[D 1], [D 2]]"
3,Sun,https://en.wikipedia.org/wiki/Sun,None
4,"[Mercury, Venus, Earth, Mars, Jupiter, Saturn,...",[https://en.wikipedia.org/wiki/Mercury_(planet...,"[None, None, None, None, None, None, None, None]"
5,"[Ceres, Orcus, Pluto, Haumea, Quaoar, Makemake...",[https://en.wikipedia.org/wiki/Ceres_(dwarf_pl...,"[None, None, None, None, None, None, None, Non..."
6,758,None,[D 3]
7,"1,462,402",None,[D 4]
8,"4,629",None,[D 4]
9,G2V,None,None


In [ ]:
url = "https://en.wikipedia.org/wiki/Solar_System"
headers = {"User-Agent": "garrettstoll.info/1.0 (garrett@garrettstoll.info)"}
response = requests.get(url, headers=headers)
soup = bs4.BeautifulSoup(response.text, 'html.parser')

infobox = getInfoBox(soup, url)
links = infobox.loc[('Population', 'Planets')]['Links']
planets = pd.DataFrame()
for i, planet in enumerate(infobox.loc[('Population', 'Planets')]['Value']):
    
    infoBox = savePageAndInfoBox(links[i], planet.lower(), 'data/Sol/planets')
    
    tmp = pd.DataFrame({('Metadata', 'PageName'): [planet] * 3})
    infoBox = pd.concat([tmp.reset_index(drop=True), infoBox.T.reset_index()], axis=1)
    infoBox.columns = pd.MultiIndex.from_tuples([('Metadata', 'RowType') if col == ('index', '') else col for col in infoBox.columns])
    #planets = pd.concat([planets, infoBox], axis=0, join='outer', ignore_index=True, sort=False)

0
1
2
3
4
5
6
7


In [64]:
earth = pd.read_csv('data/Sol/planets/earth_infoBox_wiki.csv', index_col=[0,1])

In [65]:
earth

Value  \
Category                 Label                                                                             
Designations             Alternative names             ['The world', 'The globe', 'Terra', 'Tellus', ...   
                         Adjectives                    ['Earthly', 'Terrestrial', 'Terran', 'Tellurian']   
                         Symbol                                                                      and   
Orbital characteristics  Aphelion                                                         152 097 597 km   
                         Perihelion                                                       147 098 450 km   
                         Semi-major axis                                                  149 598 023 km   
                         Eccentricity                                                         0.016 7086   
                         Orbital period (sidereal)                                     365.256 363 004 d   
                         Averageorbital speed                                               29.7827 km/s   
                         Mean anomaly                                                           358.617°   
                         Inclination                   ["7.155° – Sun 's equator;", '1.578 69 ° – inv...   
                         Longitude of ascending node                       −11.260 64 ° – J2000 ecliptic   
                         Time of perihelion                                               3 January 2026   
                         Argument of perihelion                                             114.207 83 °   
                         Satellites                                                          1, the Moon   
Physical characteristics Mean radius                                                         6 371 .0 km   
                         Equatorialradius                                                  6 378 .137 km   
                         Polarradius                                                       6 356 .752 km   
                         Flattening                                                   1/ 298.257 222 101   
                         Circumference                 ['40 075 .017\xa0km equatorial', '40 007 .86\x...   
                         Surface area                  ['510 072 000 km 2', 'Land: 148 940 000 km 2',...   
                         Volume                                                    1.083 21 × 10 12 km 3   
                         Mass                                                                 × 10 24 kg   
                         Meandensity                                                        5.513 g/cm 3   
                         Surface gravity                                                  9.806 65 m/s 2   
                         Moment of inertia factor                                                 0.3307   
                         Escape velocity                                                     11.186 km/s   
                         Synodic rotation period                                                   1.0 d   
                         Sidereal rotation period                                         0.997 269 68 d   
                         Equatorial rotation velocity                                        1674.4 km/h   
                         Axial tilt                                                        23.439 2811 °   
                         Albedo                                        ['0.434 geometric', '0.294 Bond']   
                         Temperature                                                               255 K   
                         Surfaceequivalent doserate                                          0.274 μSv/h   
                         Absolute magnitude(H)                                                     −3.99   
Atmosphere               Surfacepressure                                                     101.325 kP

In [67]:
earth.loc[('Atmosphere', 'Composition by volume')].iloc[2]

'[None, None, None, None, None, None, None, None, None, None]'